# Lesson 07 - Disenyo ng Pattern ng Pagpaplano

Ipinapakita ng notebook na ito ang **Disenyo ng Pattern ng Pagpaplano** para sa mga AI agent gamit ang Microsoft Agent Framework.
Matututo kang hatiin ang mga kumplikadong kahilingan sa paglalakbay sa mga nakaayos na subtask, italaga ang mga ito sa mga espesyalistang agent,
at isagawa ang nagresultang plano — lahat gamit ang nakaayos na output na pinapatakbo ng mga modelong Pydantic.


## Setup


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Paghahati ng Gawain

Ang paghahati ng gawain ay ang puso ng disenyo ng pattern ng pagpaplano. Sa halip na humiling sa isang ahente na hawakan ang isang komplikadong kahilingan mula simula hanggang dulo, hinahati namin ang problema sa mas maliliit at malinaw na **mga bahagi ng gawain**. Ang bawat bahagi ay itinalaga sa isang espesyalistang ahente (hal., mga flight, hotel, aktibidad) na may malinaw na prayoridad at pagkakasunod-sunod ng mga depensiya.

Ang pamamaraang ito ay nagbibigay ng ilang mga benepisyo:
- **Kalipawan**: ang bawat bahagi ng gawain ay may isang tanging responsibilidad.
- **Pagsabay-sabay**: ang mga independiyenteng bahagi ay maaaring patakbuhin ng sabay-sabay.
- **Pagkakatiwalaan**: ang mga pagkabigo ay naihihiwalay sa bawat bahagi ng gawain.
- **Pagsubaybay sa badyet**: tinatantiya ang mga gastos bawat bahagi ng gawain at pinag-iisa.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Paglikha ng Planning Agent na may Structured Output

Ang planning agent ay gumaganap bilang isang **front desk coordinator**. Kapag binigyan ng isang high-level na kahilingan sa paglalakbay, ito ay lumilikha ng isang structured na `TravelPlan` — hinahati ang kahilingan sa mga subtasks, nagtatakda ng mga prayoridad, at tinutukoy ang mga dependencies upang ang isang concierge o execution layer ay maaaring maisagawa ang gawain.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Pagsasagawa ng Plano gamit ang mga Espesyalisadong Tool

Kapag nakagawa na ang front desk agent ng isang istrukturadong plano, isinasagawa ito ng **concierge agent**. Bawat espesyalisadong tool ay humahawak ng isang kategorya ng subtask (mga flight, hotel, aktibidad). Ang concierge ay inuulit-ulit ang mga subtask ng plano ayon sa pagkakasunod-sunod ng dependencies at ipinapadala ang bawat isa sa angkop na tool.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Summary

Sa leksyong ito natutunan mo ang **Planning Design Pattern** para sa mga AI agent:

1. **Task Decomposition** — Isang front desk planning agent ang naghahati ng isang komplikadong kahilingan sa paglalakbay sa mga nakaayos na subtask gamit ang mga Pydantic models, na nagtatalaga ng bawat isa sa isang espesyalistang agent na may mga prayoridad at dependencies.
2. **Structured Output** — Sa pamamagitan ng pagpapasa ng `response_format` ang agent ay nagbabalik ng isang validated na `TravelPlan` object sa halip na malayang teksto, na nagpapadali sa maaasahang downstream processing.
3. **Plan Execution** — Isang concierge agent ang nagpapatuloy sa mga subtask gamit ang mga specialist tools (`book_flight`, `reserve_hotel`, `book_activity`) upang isagawa ang plano at iulat ang mga resulta.

Ang pattern na ito ay naghihiwalay ng *kung ano ang gagawin* (planning) mula sa *kung paano ito gagawin* (execution), na ginagawang mas modular, testable, at madaling palawakin ang mga agent.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagsang-ayon**:
Ang dokumentong ito ay isinalin gamit ang AI na serbisyo sa pagsasalin na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagaman nagsusumikap kami para sa katumpakan, pakatandaan na maaaring may mga error o kamalian ang awtomatikong pagsasalin. Ang orihinal na dokumento sa kanyang sariling wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang hindi pagkakaintindihan o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
